In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.papm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
209,NaN,2025-26,201145,Jeff Green,Jeff,1610612745,HOU,Houston Rockets,22501194,2026-04-12T00:00:00,HOU vs. MEM,W,24.183333,2,8,0.250,0,4,0.00,2,2,1.000,2,3,5,1,3,1,1,0,2,1,6,7,16.5,0,0,16.0,1,24:11,1,103.9,105.5,105.5,86.1,91.1,91.1,17.8,14.4,14.4,0.059,0.33,7.7,0.063,0.100,0.081,23.1,23.3,0.250,0.338,0.174,0.181,114.13,110.16,91.80,110.16,0.016,55,2.0,8.0,48,105,0.457,14,49,0.286,22,30,0.733,21,43,64,23,11.0,10,8,6,10,24,132,31.0,122.0,128.2,94.9,98.1,27.1,30.1,0.479,2.09,14.9,0.452,0.821,0.627,0.107,0.524,0.558,107.3,103.0,85.83,103,0.642,1610612763,MEM,Memphis Grizzlies,39,95,0.411,13,49,0.265,10,10,1.000,6,31,37,23,13.0,8,6,8,24,10,101,-31.0,94.9,98.1,122.0,128.2,-27.1,-30.1,0.590,1.77,16.9,0.179,0.548,0.373,0.126,0.479,0.508,107.3,103.0,85.83,103,0.358,NaN,PF,39.0
210,NaN,2025-26,1631342,Daeqwon Plowden,Daeqwon,1610612758,SAC,Sacramento Kings,22501200,2026-04-12T00:00:00,SAC @ POR,L,22.200000,2,5,0.400,1,4,0.25,3,3,1.000,0,1,1,2,3,0,2,0,4,1,8,-8,15.2,0,0,16.0,1,22:12,1,118.2,114.6,114.6,127.2,134.0,134.0,-9.0,-19.5,-19.5,0.100,0.67,18.2,0.000,0.034,0.022,27.3,26.5,0.500,0.633,0.173,0.184,103.83,102.70,85.59,102.70,0.017,48,2.0,5.0,40,83,0.482,7,21,0.333,23,31,0.742,15,32,47,24,17.0,7,8,3,18,21,110,-12.0,111.5,112.2,120.1,123.2,-8.6,-11.0,0.600,1.41,17.3,0.370,0.600,0.495,0.173,0.524,0.569,100.1,98.5,82.08,98,0.478,1610612757,POR,Portland Trail Blazers,47,101,0.465,16,46,0.348,12,15,0.800,18,28,46,28,12.0,9,3,8,21,18,122,12.0,120.1,123.2,111.5,112.2,8.6,11.0,0.596,2.33,18.8,0.400,0.630,0.505,0.121,0.545,0.567,100.1,98.5,82.08,99,0.522,F,SG,27.0
211,NaN,2025-26,1630286,Trevon Scott,Trevon,1610612751,BKN,Brooklyn Nets,22501192,2026-04-12T00:00:00,BKN @ TOR,L,26.050000,3,8,0.375,2,5,0.40,0,0,0.000,1,2,3,1,2,1,0,0,2,0,8,-17,14.1,0,0,16.0,1,26:03,1,104.8,109.3,109.3,137.8,135.7,135.7,-32.9,-26.5,-26.5,0.053,0.50,9.1,0.038,0.125,0.071,18.2,18.2,0.500,0.500,0.164,0.161,102.67,101.34,84.45,101.34,0.027,54,3.0,8.0,35,86,0.407,9,34,0.265,22,29,0.759,13,25,38,21,16.0,9,0,8,27,20,101,-35.0,99.3,101.0,136.3,136.0,-37.1,-35.0,0.600,1.31,15.6,0.255,0.844,0.471,0.160,0.459,0.511,100.8,100.0,83.33,100,0.277,1610612761,TOR,Toronto Raptors,51,80,0.638,12,27,0.444,22,29,0.759,3,40,43,36,10.0,9,8,0,20,27,136,35.0,136.3,136.0,99.3,101.0,37.1,35.0,0.706,3.60,25.5,0.156,0.745,0.529,0.100,0.713,0.733,100.8,100.0,83.33,100,0.723,C,NaN,NaN
204,NaN,2025-26,1629020,

### Load latest odds on file

In [5]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260416_183335.json


,home_team,away_team,commence_time,bookmakers
0,Orlando Magic,Charlotte Hornets,2026-04-17 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Phoenix Suns,Golden State Warriors,2026-04-18 02:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Cleveland Cavaliers,Toronto Raptors,2026-04-18 17:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Denver Nuggets,Minnesota Timberwolves,2026-04-18 19:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,New York Knicks,Atlanta Hawks,2026-04-18 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [6]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')
pts_ast_df = pd.read_csv('data/processed/training/S26_TRAINING_PAPM.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
lines_dfs_pts_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points_assists')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()
pts_ast_names = lines_dfs_pts_ast['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
lines_us_pts_ast = lines_us[(lines_us['CATEGORY'] == 'player_points_assists')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-04-16 18:32:44
US latest pull: 2026-04-16 18:33:35


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Betr DFS,player_points,LaMelo Ball,Over,22.5,-137,2026-04-17,2026-04-17T01:32:03Z,2026-04-16 18:32:44
1,Betr DFS,player_points,LaMelo Ball,Under,22.5,-137,2026-04-17,2026-04-17T01:32:03Z,2026-04-16 18:32:44
2,Betr DFS,player_points,Coby White,Over,13.5,-137,2026-04-17,2026-04-17T01:32:03Z,2026-04-16 18:32:44
3,Betr DFS,player_points,Coby White,Under,13.5,-137,2026-04-17,2026-04-17T01:32:03Z,2026-04-16 18:32:44
4,Betr DFS,player_points,Moussa Diabate,Over,7.5,-137,2026-04-17,2026-04-17T01:32:03Z,2026-04-16 18:32:44


### Load my models

In [7]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [8]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] Keldon Johnson: list index out of range
[SKIP] Devin Vassell: list index out of range
[SKIP] Julian Champagnie: list index out of range
[SKIP] De'Aaron Fox: list index out of range
[SKIP] Victor Wembanyama: list index out of range
[SKIP] Stephon Castle: list index out of range
[SKIP] Dylan Harper: list index out of range
[SKIP] Harrison Barnes: list index out of range
[SKIP] Jamal Cain: single positional indexer is out-of-bounds
[SKIP] Victor Wembanyama: list index out of range
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Julian Champagnie: list index out of range
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,LaMelo Ball,PTS,15.80,29.67,37.33,0.4532,0.6966,0.9548,7.16,20.67,35.64,"[0.9797517962116264, 0.7762557077625571, 0.905..."
1,Coby White,PTS,11.60,19.47,30.00,0.2754,0.6058,0.9809,3.19,11.80,29.43,"[0.3573251659009699, 0.4522613065326633, 0.687..."
2,Moussa Diabaté,PTS,14.98,24.37,32.83,0.1141,0.2893,0.5330,1.71,7.05,17.50,"[0.2277163305139883, 0.0588581518540317, 0.187..."
3,Brandon Miller,PTS,14.45,28.97,37.75,0.3519,0.5743,0.8511,5.09,16.64,32.13,"[0.6103143118706134, 0.1820940819423368, 0.607..."
4,Miles Bridges,PTS,15.30,27.11,36.38,0.2511,0.4930,0.7861,3.84,13.37,28.60,"[0.8312020460358056, 0.6505026611472502, 0.445..."
5,Kon Knueppel,PTS,19.95,28.92,36.49,0.2595,0.4874,0.7825,5.18,14.09,28.56,"[0.699912510936133, 0.6323110970597534, 0.9314..."
6,Franz Wagner,PTS,11.47,23.99,33.01,0.3101,0.6224,0.8450,3.56,14.93,27.89,"[0.7885843034171987, 0.5838198498748958, 0.773..."
7,Jalen Suggs,PTS,14.53,27.07,35.32,0.1975,0.4374,0.7061,2.87,11.84,24.94,"[0.8218373936014088, 0.4095962551199532, 0.281..."
8,Paolo Banchero,PTS,12.30,28.55,38.60,0.3794,0.6092,0.8467,4.67,17.39,32.69,"[0.5394066526820498, 0.883489784649365, 0.7047..."
9,Desmond Bane,PTS,13.18,27.51,36.37,0.3286,0.5430,0.7703,4.33,14.94,28.02,"[0.576923076923077, 0.4411359250068927, 0.9327..."


### Get Line Probabilities

In [9]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
159,Aaron Gordon,PTS,14.5,14.61,24.09,32.37,3.70,11.54,24.20,0.353,0.647
166,Nickeil Alexander-Walker,PTS,20.5,15.24,28.73,37.22,3.85,15.04,28.59,0.383,0.618
100,Neemias Queta,REB,7.5,10.42,20.84,28.71,1.78,6.64,13.60,0.468,0.532
101,Deni Avdija,REB,6.5,18.07,33.48,40.50,1.51,6.50,12.67,0.437,0.563
134,Brandin Podziemski,PTS,14.5,15.38,29.59,37.71,3.43,13.99,28.71,0.700,0.300
3,Desmond Bane,AST,3.5,13.18,27.51,36.37,0.47,3.83,8.34,0.480,0.520
43,Anthony Black,AST,2.5,8.75,19.00,27.39,0.19,2.37,6.20,0.365,0.635
94,Tyrese Maxey,REB,3.5,19.35,34.58,41.51,0.33,3.29,8.26,0.457,0.543
45,LaMelo Ball,REB,5.5,15.80,29.67,37.33,0.80,4.28,9.65,0.365,0.635
197,Deni Avdija,PTS,23.5,18.07,33.48,40.50,6.59,20.85,35.58,0.520,0.480


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
178,Amen Thompson,PTS,18.5,14.23,30.58,39.56,3.01,16.67,30.45,0.379,0.621,PTS,Underdog,Los Angeles Lakers,-5.5,207.5,115.5,20.0,99.22,22.0,-107.0,-112.0,0.517,0.528,19.9,18.0,8.06,1.4,-0.5,-0.174,0.569,0.431,10.08,-18.42,0.8,0.4,0.53,0.38,38.87,3.82,0.19,0.05,22.80,5.0
114,Marcus Smart,REB,2.5,15.11,28.55,36.59,0.01,2.59,6.98,0.467,0.533,REB,Underdog,Houston Rockets,5.5,207.5,112.1,6.0,96.98,29.0,-105.0,100.0,0.512,0.500,2.4,2.5,1.35,-0.1,0.0,0.074,0.471,0.529,-8.04,5.80,0.4,0.5,0.53,0.49,29.71,5.21,0.14,0.05,1.50,4.0
0,LaMelo Ball,AST,7.5,15.80,29.67,37.33,0.63,4.49,9.56,0.343,0.657,AST,Underdog,Orlando Magic,-3.5,218.0,113.6,13.0,100.56,14.0,-107.0,-110.0,0.517,0.524,7.4,8.0,2.22,-0.1,0.5,0.045,0.482,0.518,-6.75,-1.11,0.6,0.6,0.53,0.45,31.16,4.27,0.30,0.06,7.57,7.0
134,Brandin Podziemski,PTS,14.5,15.38,29.59,37.71,3.43,13.99,28.71,0.700,0.300,PTS,Underdog,Phoenix Suns,3.5,219.5,112.9,9.0,98.14,24.0,-120.0,105.0,0.545,0.488,20.1,21.0,5.69,6.6,7.5,-1.160,0.877,0.123,60.78,-74.78,0.8,0.9,0.73,0.45,29.62,6.63,0.24,0.04,11.25,8.0
198,Jrue Holiday,PTS,15.5,16.37,29.17,38.15,4.30,14.37,28.63,0.415,0.585,PTS,Underdog,San Antonio Spurs,10.8,222.2,110.4,3.0,100.72,12.0,-116.0,-107.0,0.537,0.517,17.9,16.0,7.49,2.4,0.5,-0.320,0.626,0.374,16.57,-27.65,0.6,0.5,0.33,0.35,31.38,5.24,0.23,0.07,17.00,2.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
204,Grant Williams,PTS,5.5,13.38,20.63,28.15,0.67,6.52,16.53,0.667,0.333,PTS,PrizePicks,Orlando Magic,-3.5,218.0,113.6,13.0,100.56,14.0,-106.0,-114.0,0.515,0.533,6.6,8.0,3.47,1.1,2.5,-0.317,0.624,0.376,21.27,-29.42,0.8,0.6,0.60,0.69,20.39,3.37,0.12,0.06,7.50,2.0
0,LaMelo Ball,AST,7.5,15.80,29.67,37.33,0.63,4.49,9.56,0.343,0.657,AST,PrizePicks,Orlando Magic,-3.5,218.0,113.6,13.0,100.56,14.0,-107.0,-110.0,0.517,0.524,7.4,8.0,2.22,-0.1,0.5,0.045,0.482,0.518,-6.75,-1.11,0.6,0.6,0.53,0.45,31.16,4.27,0.30,0.06,7.57,7.0
65,Donovan Mitchell,REB,4.0,13.09,28.47,35.42,0.60,3.58,8.19,0.516,0.484,REB,PrizePicks,Toronto Raptors,-8.2,219.5,112.1,5.0,99.22,21.0,-137.0,-137.0,0.578,0.578,5.1,6.0,2.18,1.1,2.0,-0.505,0.693,0.307,19.88,-46.89,0.6,0.6,0.53,0.43,33.33,3.87,0.29,0.07,3.80,5.0
169,Jonathan Kuminga,PTS,10.0,15.24,23.27,31.23,3.93,12.45,25.71,0.694,0.306,PTS,PrizePicks,New York Knicks,5.5,216.5,112.3,7.0,97.71,25.0,-137.0,-137.0,0.578,0.578,10.7,11.0,6.68,0.7,1.0,-0.105,0.542,0.458,-6.24,-20.77,0.8,0.5,0.53,0.63,21.81,3.59,0.20,0.05,7.50,2.0
59,Jakob Poeltl,REB,7.5,12.95,23.94,30.63,1.64,6.62,14.41,0.280,0.720,REB,PrizePicks,Cleveland Cavaliers,8.2,219.5,114.1,15.0,100.70,13.0,110.0,-115.0,0.476,0.535,5.2,5.0,2.74,-2.3,-2.5,0.839,0.201,0.799,-57.79,49.38,0.0,0.1,0.27,0.60,23.09,4.06,0.18,0.05,11.00,5.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
17,Jalen Brunson,AST,7.0,12.19,28.84,36.97,0.47,4.59,11.12,0.379,0.621,AST,Betr DFS,Atlanta Hawks,-5.5,216.5,112.9,10.0,102.50,5.0,-137.0,-137.0,0.578,0.578,8.0,8.0,3.65,1.0,1.0,-0.274,0.608,0.392,5.18,-32.19,0.8,0.6,0.67,0.42,35.86,4.40,0.29,0.05,7.00,7.0
86,Karl-Anthony Towns,REB,12.0,13.01,25.22,32.97,2.36,8.29,15.84,0.258,0.742,REB,Betr DFS,Atlanta Hawks,-5.5,216.5,112.9,10.0,102.50,5.0,-137.0,-137.0,0.578,0.578,11.9,12.0,4.31,-0.1,0.0,0.023,0.491,0.509,-15.06,-11.95,0.0,0.4,0.33,0.43,29.06,4.04,0.27,0.06,13.14,7.0
69,Naz Reid,REB,6.0,14.40,23.79,30.91,1.64,5.21,11.94,0.373,0.627,REB,Betr DFS,Denver Nuggets,6.0,231.5,116.0,21.0,99.49,20.0,-137.0,-137.0,0.578,0.578,5.9,5.5,3.03,-0.1,-0.5,0.033,0.487,0.513,-15.75,-11.25,0.4,0.4,0.33,0.40,25.55,3.76,0.23,0.03,5.00,7.0
27,Jayson Tatum,AST,5.0,12.23,27.71,38.12,0.43,4.18,10.76,0.485,0.514,AST,Betr DFS,Philadelphia 76ers,-12.5,213.5,114.4,17.0,100.40,15.0,-137.0,-137.0,0.578,0.578,6.1,7.0,3.35,1.1,2.0,-0.328,0.629,0.371,8.81,-35.82,0.8,0.6,0.47,0.53,34.54,3.55,0.29,0.06,6.75,4.0
64,Jarrett Allen,REB,9.0,12.64,23.08,29.75,2.17,7.17,13.44,0.352,0.648,REB,Betr DFS,Toronto Raptors,-8.2,219.5,112.1,5.0,99.22,21.0,-137.0,-137.0,0.578,0.578,8.3,9.0,3.13,-0.7,0.0,0.224,0.411,0.589,-28.90,1.89,0.4,0.4,0.53,0.50,25.96,5.49,0.23,0.04,10.60,5.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
55,Gui Santos,REB,5.5,13.66,24.02,32.64,0.79,4.00,10.58,0.308,0.692,REB,DraftKings Pick6,Phoenix Suns,3.5,219.5,112.9,9.0,98.14,24.0,128.0,-135.0,0.439,0.574,4.8,4.0,2.35,-0.7,-1.5,0.298,0.383,0.617,-12.68,7.40,0.4,0.3,0.27,0.23,29.62,5.80,0.22,0.06,3.67,6.0
49,Franz Wagner,REB,4.5,11.47,23.99,33.01,0.57,3.29,8.08,0.331,0.669,REB,DraftKings Pick6,Charlotte Hornets,3.5,218.0,113.5,11.0,97.60,26.0,114.0,-134.0,0.467,0.573,3.1,3.0,2.73,-1.4,-1.5,0.513,0.304,0.696,-34.94,21.54,0.2,0.2,0.33,0.67,21.78,3.61,0.27,0.05,5.80,5.0
132,Draymond Green,PTS,8.5,14.46,27.13,35.62,1.63,7.95,18.63,0.369,0.631,PTS,DraftKings Pick6,Phoenix Suns,3.5,219.5,112.9,9.0,98.14,24.0,-108.0,-112.0,0.519,0.528,7.6,7.0,4.22,-0.9,-1.5,0.213,0.416,0.584,-19.88,10.54,0.0,0.3,0.40,0.46,29.65,6.07,0.13,0.04,7.43,7.0
105,Miles Bridges,REB,6.5,15.30,27.11,36.38,1.25,5.20,11.65,0.426,0.574,REB,DraftKings Pick6,Orlando Magic,-3.5,218.0,113.6,13.0,100.56,14.0,115.0,-142.0,0.465,0.587,5.7,5.5,3.16,-0.8,-1.0,0.253,0.400,0.600,-14.00,2.25,0.6,0.3,0.33,0.49,27.99,3.74,0.21,0.06,6.83,6.0
123,Kon Knueppel,PTS,16.5,19.95,28.92,36.49,5.18,14.09,28.56,0.352,0.648,PTS,DraftKings Pick6,Orlando Magic,-3.5,218.0,113.6,13.0,100.56,14.0,-112.0,-108.0,0.528,0.519,14.6,13.0,5.62,-1.9,-3.5,0.338,0.368,0.632,-30.34,21.72,0.2,0.3,0.40,0.59,31.95,3.66,0.21,0.02,12.75,4.0


In [14]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
69,Naz Reid,REB,6.0,14.40,23.79,30.91,1.64,5.21,11.94,0.373,0.627,REB,PrizePicks,Denver Nuggets,6.0,231.5,116.0,21.0,99.49,20.0,-137.0,-137.0,0.578,0.578,5.9,5.5,3.03,-0.1,-0.5,0.033,0.487,0.513,-15.75,-11.25,0.4,0.4,0.33,0.40,25.55,3.76,0.23,0.03,5.00,7.0
121,Brandon Miller,PTS,20.5,14.45,28.97,37.75,5.09,16.64,32.13,0.375,0.625,PTS,PrizePicks,Orlando Magic,-3.5,218.0,113.6,13.0,100.56,14.0,-110.0,-109.0,0.524,0.522,19.5,20.5,6.15,-1.0,0.0,0.163,0.435,0.565,-16.95,8.33,0.4,0.5,0.47,0.50,31.45,2.33,0.23,0.03,17.60,5.0
34,Jaden McDaniels,AST,2.5,13.26,26.79,34.29,0.27,3.36,8.16,0.433,0.567,AST,Underdog,Denver Nuggets,6.0,231.5,116.0,21.0,99.49,20.0,-115.0,-110.0,0.535,0.524,2.3,2.5,1.16,-0.2,0.0,0.172,0.432,0.568,-19.23,8.44,0.4,0.5,0.47,0.41,30.42,7.83,0.21,0.08,2.12,8.0
135,Gui Santos,PTS,12.5,13.66,24.02,32.64,2.90,10.50,22.15,0.448,0.552,PTS,Underdog,Phoenix Suns,3.5,219.5,112.9,9.0,98.14,24.0,-105.0,-110.0,0.512,0.524,15.2,14.0,9.65,2.7,1.5,-0.280,0.610,0.390,19.10,-25.55,0.4,0.6,0.73,0.27,29.62,5.80,0.22,0.06,6.83,6.0
159,Aaron Gordon,PTS,14.5,14.61,24.09,32.37,3.70,11.54,24.20,0.353,0.647,PTS,Underdog,Minnesota Timberwolves,-6.0,231.5,112.5,8.0,101.50,10.0,-115.0,-108.0,0.535,0.519,15.0,15.5,6.62,0.5,1.0,-0.076,0.530,0.470,-0.91,-9.48,0.6,0.6,0.53,0.52,29.05,6.20,0.19,0.05,21.20,5.0


### Get top EVs for 2 legs

In [15]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 159  |  Pairs: 772  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 70  |  Pairs: 125  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 37  |  Pairs: 28  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 153  |  Pairs: 837  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 159  |  Triples: 28925  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 70  |  Triples: 1812  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 153  |  Triples: 29576  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 37  |  Triples: 124  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
